In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import tifffile
import glob
import natsort
import shutil
import os

In [2]:
# path = '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena'
path = '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena'
train_path = os.path.join(path,'training_data')
os.makedirs(train_path,exist_ok=True)

In [3]:
def merge_tiffs(brightfield,cy5):
	c1 = tifffile.imread(brightfield)
	c2 = tifffile.imread(cy5)
	merged = np.stack([c1,c2],axis=0)
	return merged


In [4]:
dirs = glob.glob(os.path.join(path,'*','tif'))
for d in dirs:
	brightfields = glob.glob(d+'/*.brightfield.tiff')
	cy5s = glob.glob(d+'/*.cy5.tiff')
	brightfields = natsort.natsorted(brightfields)
	cy5 = natsort.natsorted(cy5s)
	
	for b,c in zip(brightfields,cy5):
		merged = merge_tiffs(b,c)
		tifffile.imwrite(b.replace('.brightfield.tiff','.merged.tiff'),merged)
		print('saved',b.replace('.brightfield.tiff','.merged.tiff'))


saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0000.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0001.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0002.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0003.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0004.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0005.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0006.merged.tiff
saved /Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0007.merged.tiff


In [5]:
for f in glob.glob(os.path.join(path,'*','tif','*_seg.npy')):
	mask_file = f.replace('_seg.npy','_mask.tiff')
	seg = np.load(f,allow_pickle=True).item()
	mask = seg['masks']
	tifffile.imwrite(mask_file,mask)
	print(f)

	dirname = f.split('/')[-3]
	filename = dirname+'.'+os.path.basename(f).replace('brightfield','merged').replace('_seg.npy','_mask.tiff')
	shutil.copyfile(mask_file,os.path.join(train_path,filename))

/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0002.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0004.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0003.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0000.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.007/tif/set1.007_0001.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.009/tif/set1.009_0000.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.009/tif/set1.009_0004.brightfield_seg.npy
/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241114_ZMB_Anabaena/set1.009/tif/set1.009_0007.brig

In [6]:
for f in glob.glob(os.path.join(path,'*','tif','*.merged.tiff')):
	dirname = f.split('/')[-3]
	filename = dirname+'.'+os.path.basename(f).replace('merged','merged_img')
	
	# Conditionally add to training data if the mask exists
	if os.path.exists(os.path.join(train_path,filename).replace('_img','_mask')):
		shutil.copyfile(f,os.path.join(train_path,filename))
